In [1]:
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
URL = "https://bankofindia.co.in/interest-rate/rupee-term-deposit-rate"

date1 = (date.today()).strftime('%d%m%Y') # Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' # Set directory path to write file

r=requests.get(URL) # Fetch the HTML page
#print(r)
soup=BeautifulSoup(r.text,"html.parser")
table=soup.find("table",class_="table")
rows=table.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

list_of_interest_rates = []
row_count=0
for i in rows[1:]:  # Skip heading
    data = i.find_all("td")
    row=['Bank of India']
    row.extend([tr.text for tr in data[:2]]) # Extract the text from the first two columns only
    if row_count<=4: #for the first 5 rows, repeat general rate as Sr citizen and Super Sr. citizen rate
        new_row_items = ['',row[2] + '%','',row[2] + '%','',30000000] 
    elif row_count<=12: # for 6th to 12th row (6mths to 3yrs), add 0.5% and 0.65% to Sr citizen and Super Sr. citizen rate
        new_row_items = ['',str(float(row[2])+0.50) + '%','',str(float(row[2])+0.65) + '%','',30000000] 
    else: # beyond 12th row (more than 3yrs), add 0.75% and 0.9% to Sr citizen and Super Sr. citizen rate
        new_row_items = ['',str(float(row[2])+0.75) + '%','',str(float(row[2])+0.9) + '%','',30000000]
    row.extend(new_row_items)
    row[2] = row[2] + '%' # Add % sign to general rate of interest
    row_count+=1
    list_of_interest_rates.append(row)
           
#pd.set_option('display.max_columns',None)
int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                    'sr_rate','annualised_sr_rate','super_sr_rate','annualised_super_sr_rate','threshold_amt'])
#print(list_of_interest_rates)
#print(int_rate_df)
int_rate_df.to_csv(file_name,mode='w',header=True,index=False) # Write the interest rate rows to the file
print("\nBank of India = Success. Number of rows added = ", len(list_of_interest_rates))

Today's date = 17112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_17112024.csv

Bank of India = Success. Number of rows added =  16
